In [ ]:
# ============================================
# Basic Small ResNet
# ============================================

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, utils, metrics
from sklearn.metrics import confusion_matrix, classification_report
     

# ============================================
# Install and Setup Kaggle
# ============================================

!pip install -q kaggle

from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d puneet6060/intel-image-classification
!unzip -q intel-image-classification.zip
     

# ============================================
# Load Training Dataset
# ============================================

train_ds = utils.image_dataset_from_directory("/content/seg_train/seg_train", image_size = (128, 128), batch_size = 32, seed = 123)

class_names = train_ds.class_names

num_batches = len(train_ds)

test_size = int(0.1 * num_batches)

# Split dataset
test_ds = train_ds.take(test_size)
train_ds = train_ds.skip(test_size)

# Optimize pipeline
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)
     

# ============================================
# Load Validation Dataset
# ============================================

val_ds = utils.image_dataset_from_directory("/content/seg_test/seg_test", image_size = (128, 128), batch_size = 32, shuffle = False)

val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
     

# ============================================
# Residual Block
# ============================================

def residual_block(x, filters):

    shortcut = x

    x = layers.Conv2D(filters, (3, 3), padding = 'same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(filters, (3, 3), padding = 'same')(x)
    x = layers.BatchNormalization()(x)

    # Match channels if needed
    if (shortcut[-1].shape != filters):
        shortcut = layers.Conv2D(filters, (1, 1), padding = 'same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)

    return x
     

# ============================================
# MODEL 1: WITH DATA AUG + DROPOUT
# ============================================

def conv_model(input_shape):

     inputs = layers.Input(input_shape)

     x = layers.RandomFlip("horizontal")(inputs)
     x = layers.RandomRotation(0.1)(x)
     x = layers.RandomZoom(0.1)(x)

     x = layers.Rescaling(1./255)(x)

     x = layers.Conv2D(32, (3, 3), padding = 'same')(x)
     x = layers.BatchNormalization()(x)
     x = layers.Activation('relu')(x)

     x = residual_block(x, 32)
     x = residual_block(x, 32)

     x = layers.MaxPooling2D((2, 2))(x)

     x = residual_block(x, 64)
     x = residual_block(x, 64)

     x = layers.MaxPooling2D((2, 2))(x)

     x = residual_block(x, 128)
     x = residual_block(x, 128)

     x = layers.GlobalAveragePooling2D()(x)

     x = layers.Dropout(0.3)(x)

     outputs = layers.Dense(6, activation='softmax')(x)

     model = models.Model(inputs , outputs)

     return model
     

resnet_model = conv_model((128, 128, 3))

resnet_model.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

history = resnet_model.fit(train_ds, epochs = 20, validation_data = val_ds)
     

# Plot
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title("Resnet Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend(['train', 'validation'])
plt.show()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title("Resnet Model Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend(['train', 'validation'])
plt.show()
     

# Prediction + Evaluation
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = resnet_model.predict(images)
    preds = np.argmax(preds, axis=1)

    y_pred.extend(preds)
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(len(y_true), len(y_pred))
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))


# ============================================
# MODEL 2: WITHOUT DATA AUGMENTATION
# ============================================

def conv_model(input_shape):

     inputs = layers.Input(input_shape)

     # No augmentation
     # x = layers.RandomFlip("horizontal")(inputs)
     # x = layers.RandomRotation(0.1)(x)
     # x = layers.RandomZoom(0.1)(x)

     x = layers.Rescaling(1./255)(inputs)

     x = layers.Conv2D(32, (3, 3), padding = 'same')(x)
     x = layers.BatchNormalization()(x)
     x = layers.Activation('relu')(x)

     x = residual_block(x, 32)
     x = residual_block(x, 32)

     x = layers.MaxPooling2D((2, 2))(x)

     x = residual_block(x, 64)
     x = residual_block(x, 64)

     x = layers.MaxPooling2D((2, 2))(x)

     x = residual_block(x, 128)
     x = residual_block(x, 128)

     x = layers.GlobalAveragePooling2D()(x)

     x = layers.Dropout(0.3)(x)

     outputs = layers.Dense(6, activation='softmax')(x)

     model = models.Model(inputs , outputs)

     return model
     

resnet_no_data_aug = conv_model((128, 128, 3))

resnet_no_data_aug.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

history1 = resnet_no_data_aug.fit(train_ds, epochs = 20, validation_data = val_ds)
     

# Prediction + Evaluation
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = resnet_no_data_aug.predict(images)
    preds = np.argmax(preds, axis=1)

    y_pred.extend(preds)
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))


# ============================================
# MODEL 3: WITHOUT DROPOUT
# ============================================

def conv_model(input_shape):

     inputs = layers.Input(input_shape)

     x = layers.RandomFlip("horizontal")(inputs)
     x = layers.RandomRotation(0.1)(x)
     x = layers.RandomZoom(0.1)(x)

     x = layers.Rescaling(1./255)(x)

     x = layers.Conv2D(32, (3, 3), padding = 'same')(x)
     x = layers.BatchNormalization()(x)
     x = layers.Activation('relu')(x)

     x = residual_block(x, 32)
     x = residual_block(x, 32)

     x = layers.MaxPooling2D((2, 2))(x)

     x = residual_block(x, 64)
     x = residual_block(x, 64)

     x = layers.MaxPooling2D((2, 2))(x)

     x = residual_block(x, 128)
     x = residual_block(x, 128)

     x = layers.GlobalAveragePooling2D()(x)

     # Dropout removed
     # x = layers.Dropout(0.3)(x)

     outputs = layers.Dense(6, activation='softmax')(x)

     model = models.Model(inputs , outputs)

     return model
     

resnet_no_dropout = conv_model((128, 128, 3))

resnet_no_dropout.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

history2 = resnet_no_dropout.fit(train_ds, epochs = 20, validation_data = val_ds)
     

# Prediction + Evaluation
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = resnet_no_dropout.predict(images)
    preds = np.argmax(preds, axis=1)

    y_pred.extend(preds)
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))


# ============================================
# MODEL 4: DIFFERENT LEARNING RATE
# ============================================

def conv_model(input_shape):

     inputs = layers.Input(input_shape)

     x = layers.RandomFlip("horizontal")(inputs)
     x = layers.RandomRotation(0.1)(x)
     x = layers.RandomZoom(0.1)(x)

     x = layers.Rescaling(1./255)(x)

     x = layers.Conv2D(32, (3, 3), padding = 'same')(x)
     x = layers.BatchNormalization()(x)
     x = layers.Activation('relu')(x)

     x = residual_block(x, 32)
     x = residual_block(x, 32)

     x = layers.MaxPooling2D((2, 2))(x)

     x = residual_block(x, 64)
     x = residual_block(x, 64)

     x = layers.MaxPooling2D((2, 2))(x)

     x = residual_block(x, 128)
     x = residual_block(x, 128)

     x = layers.GlobalAveragePooling2D()(x)

     x = layers.Dropout(0.3)(x)

     outputs = layers.Dense(6, activation='softmax')(x)

     model = models.Model(inputs , outputs)

     return model
     

resnet_diff_learning_rate = conv_model((128, 128, 3))

resnet_diff_learning_rate.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-4),
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

history3 = resnet_diff_learning_rate.fit(train_ds, epochs = 20, validation_data = val_ds)
     

# Prediction + Evaluation
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = resnet_diff_learning_rate.predict(images)
    preds = np.argmax(preds, axis=1)

    y_pred.extend(preds)
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))